# 2. Fine-tune Llama 3 8B with QLoRA

**Outcome:** compare majority, TF-IDF, original Llama 3, and a trained adapter on identical held-out examples; save your adapter and measured reports.

Before running:
1. Request access to [Meta-Llama-3-8B](https://huggingface.co/meta-llama/Meta-Llama-3-8B), accept the license, and wait for approval.
2. Create a Hugging Face read token with permission for gated repositories. Store it in Colab Secrets as `HF_TOKEN` and enable notebook access.
3. Select **Runtime → Change runtime type → GPU**. This configuration targets a single NVIDIA GPU with roughly 16 GB or more VRAM. T4 compatibility is a target, not a verified guarantee. Colab may not allocate a free GPU.
4. Allow at least 25 GiB free disk after environment installation. Installation/model download may take several minutes. Do not buy compute based on an unverified runtime estimate.

The repo uses the requested **base Llama 3 8B**, not an Instruct model. It therefore uses one explicit completion prompt throughout, with no chat-template assumptions.

[Open this lesson in Colab](https://colab.research.google.com/github/chenghungpan/llama3-qlora-finance-colab/blob/main/notebooks/02_qlora_training.ipynb) · [GitHub repository](https://github.com/chenghungpan/llama3-qlora-finance-colab)


In [ ]:
from pathlib import Path
import os, subprocess, sys, zipfile

# The public repository is downloaded automatically. Clear this URL to upload a project ZIP instead.
REPO_URL = "https://github.com/chenghungpan/llama3-qlora-finance-colab.git"
PROJECT_NAME = "llama3-qlora-finance-colab"
if Path("finance_qlora/core.py").exists():
    ROOT = Path.cwd()
else:
    parent = Path("/content") if Path("/content").exists() else Path.cwd()
    ROOT = parent / PROJECT_NAME
    if not ROOT.exists():
        if REPO_URL:
            subprocess.run(["git", "clone", REPO_URL, str(ROOT)], check=True)
        else:
            from google.colab import files
            print("Upload llama3-qlora-finance-colab.zip (the whole project, not just a notebook).")
            uploaded = files.upload()
            archive = next((name for name in uploaded if name.endswith(".zip")), None)
            if archive is None:
                raise ValueError("Please upload the project ZIP")
            with zipfile.ZipFile(archive) as z:
                for member in z.infolist():
                    target = (parent / member.filename).resolve()
                    if not target.is_relative_to(parent.resolve()):
                        raise ValueError("Unsafe archive path")
                z.extractall(parent)
    if not (ROOT / "finance_qlora/core.py").exists():
        raise ValueError("Expected the standalone project directory in the ZIP")
    os.chdir(ROOT)
print("Project directory:", ROOT)


## Install the pinned environment

Use a fresh Colab runtime. The next cell removes unused vision/audio packages to avoid conflicts with pinned Torch. Subsequent ML jobs run in fresh subprocesses. This does not modify your local computer.


In [ ]:
# Only run in a disposable Colab runtime or a dedicated environment.
# Vision/audio libraries are unused and may conflict with our pinned Torch version.
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchvision", "torchaudio"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt", "-e", "."], check=True)
# ML commands below launch fresh Python processes, so they see the installed versions.


In [ ]:
# In Colab's left sidebar, open Secrets (key icon), add HF_TOKEN, and enable notebook access.
# Never paste a token into a cell or print it.
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
subprocess.run([sys.executable, "-m", "finance_qlora.preflight"], check=True)


## Choose a run and persistent storage

Start with `smoke`: 3 optimizer steps verify wiring, not learning. `quick` trains on 1,500 examples for one epoch; `full` uses all cleaned training data for two epochs. Smoke/quick use balanced subsets, so their metrics do not estimate natural class prevalence.

Set `SAVE_TO_DRIVE=True` to survive runtime loss. Checkpoints include optimizer state and can consume more storage than an adapter. To resume, keep the same MODE and RUN_NAME, set RESUME=True, rerun setup/auth/data, then rerun training. If the session died before the first checkpoint, use a new RUN_NAME. The smoke run is shorter than the checkpoint interval.


In [ ]:
MODE = "smoke"  # "smoke", "quick", or "full"
RUN_NAME = "smoke-01"  # Change for every new experiment.
SAVE_TO_DRIVE = True
RESUME = False
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    RUN_DIR = Path("/content/drive/MyDrive/llama3-finance-runs") / RUN_NAME
else:
    RUN_DIR = ROOT / "runs" / RUN_NAME
CONFIG = ROOT / "configs" / f"{MODE}.json"
assert CONFIG.exists(), "Unknown MODE"
print(CONFIG.read_text())
print("Outputs:", RUN_DIR)


In [ ]:
subprocess.run([sys.executable, "-m", "finance_qlora.data"], check=True)


## What QLoRA changes

The pretrained base weights are frozen and stored using 4-bit NF4 quantization with double quantization. Small low-rank matrices are trained on the linear layers: `W(x) + scale × B(A(x))`. Rank 8 controls adapter capacity; alpha 16 controls scaling. The optimizer updates adapter weights, not all 8 billion base parameters.

We use batch size 1, gradient accumulation 8, length 256, gradient checkpointing, and an 8-bit paged optimizer to reduce memory. BF16 is selected when supported, otherwise FP16 (e.g. T4).

Only answer tokens and the real EOS token contribute to training loss. Prompt tokens and padding are masked with -100. Long text is truncated while preserving the instruction, answer cue, and label. We use Transformers Trainer directly so this masking is easy to inspect in `core.py` and `modeling.py`.

Validation loss is monitored during training; the final adapter is evaluated. The code does not select a checkpoint or tune hyperparameters using test scores. The original model is evaluated before adding adapters, so only one base model occupies GPU memory.


In [ ]:
command = [sys.executable, "-m", "finance_qlora.run", "--config", str(CONFIG), "--output", str(RUN_DIR)]
if RESUME:
    command.append("--resume")
subprocess.run(command, check=True)


## Read the evidence

Invalid outputs are errors, even if a valid class appears somewhere in a longer answer. Do not assume fine-tuning wins. A simple classifier may outperform Llama on this dataset. Public historical examples may have appeared in pretraining, so this is not a clean test of unseen-world knowledge.


In [ ]:
from IPython.display import display, Markdown, Image
display(Markdown((RUN_DIR / "report.md").read_text()))
display(Image(filename=str(RUN_DIR / "comparison.png")))
display(Image(filename=str(RUN_DIR / "confusion.png")))


## Save and inspect

Your run directory contains the adapter, tokenizer, run config, exact selected IDs, dataset checksums, package versions, metrics, generation token counts/latencies, validation losses, and compute time. A dollar estimate is unknown by default; the CLI accepts `--hourly-usd` only as an explicit user assumption.

Download this ZIP as a backup, or leave the files in Drive. It excludes training checkpoints to reduce archive size. The adapter alone cannot run: lesson 3 reloads the pinned base weights plus the adapter. No model or data is published automatically.


In [ ]:
import zipfile
archive = ROOT / f"{RUN_NAME}-artifacts.zip"
with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for p in RUN_DIR.rglob("*"):
        if p.is_file() and "checkpoints" not in p.relative_to(RUN_DIR).parts:
            z.write(p, arcname=str(Path(RUN_NAME) / p.relative_to(RUN_DIR)))
from google.colab import files
files.download(str(archive))


## Next experiment

After smoke succeeds, use `MODE="quick"` and a new RUN_NAME. For hyperparameter comparisons, use validation and reserve a fresh final test when needed. Never repeatedly tune to the fixed test scores. Disconnect the GPU when finished using **Runtime → Disconnect and delete runtime**. Continue with lesson 3 to prove your adapter reloads in a fresh session.
